# Notebook 01 — Data Collection
## NBA Contract Value Index (CVI)

This notebook is responsible for collecting and cleaning all player performance 
data from Basketball-Reference. The output is a clean master CSV that feeds 
into every subsequent notebook in the project.

---

### What This Notebook Does

| Step | Action | Output |
|------|--------|--------|
| 1 | Define pull functions for BBRef per-game stats | `pull_bbref_stats()` |
| 2 | Define pull functions for BBRef advanced stats | `pull_bbref_advanced()` |
| 3 | Pull all seasons (2021-22 through 2025-26) | Raw DataFrames |
| 4 | Handle traded players — keep TOT rows, drop team splits | Clean DataFrames |
| 5 | Merge per-game + advanced stats | Single merged DataFrame |
| 6 | Filter to qualifying players (20+ games) | Filtered DataFrame |
| 7 | Fix broken unicode player names | Clean name column |
| 8 | Standardize column names and position groups | Final master table |
| 9 | Save to CSV | `data/raw/bbref_master.csv` |

---

### Data Source
- **Basketball-Reference** (basketball-reference.com)
- Per-game stats: points, rebounds, assists, steals, blocks, turnovers, minutes
- Advanced stats: BPM, OBPM, DBPM, VORP, WS, WS/48, PER, TS%, USG%

---

### Output
- `data/raw/bbref_master.csv`
- One row per player-season
- 5 seasons: 2021-22 through 2025-26
- ~2663 player-seasons after filtering

---

### Key Decisions Made
- Used `requests.get()` with browser headers to bypass BBRef scraping blocks
- Kept TOT rows for traded players (full season totals) and dropped team splits
- Captured LAST_TEAM separately before deduplication for market/apron features
- Standardized position to G / F / C groups for age curve calculations
- Fixed unicode encoding issues manually (Jokic, Doncic, etc.)
- Filtered to 20+ games played to remove 10-day contracts and cup-of-coffee appearances

---

### Notes for V2
- Add NBA API data for NET_RATING, PIE, play type breakdowns
- Add hustle stats (contested shots, deflections, charges drawn)
- Expand to more historical seasons (2015-16 onward) for larger training set

In [88]:
import pandas as pd
import numpy as np
import time
import warnings
import os
import unicodedata
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)


print("✓ All imports loaded")

✓ All imports loaded


# BBall Reference Stats Pull

## Define Pull Functions

In [63]:
def pull_bbref_stats(season_end_year):
    url = f"https://www.basketball-reference.com/leagues/NBA_{season_end_year}_per_game.html"
    print(f"Pulling per-game {season_end_year-1}-{str(season_end_year)[-2:]}...")
    
    headers = {"User-Agent": "Mozilla/5.0"}
    resp = requests.get(url, headers=headers)
    tables = pd.read_html(resp.text)
    df = tables[0]
    
    df = df[df["Player"] != "Player"].copy()
    df = df[df["Rk"].notna()].copy()
    df["SEASON"] = f"{season_end_year-1}-{str(season_end_year)[-2:]}"
    
    skip_cols = ["Player", "Team", "Pos", "Awards", "SEASON"]
    for col in df.columns:
        if col not in skip_cols:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    
    print(f"  ✓ {len(df)} rows — Team sample: {df['Team'].dropna().head(3).tolist()}")
    time.sleep(4)
    return df

def pull_bbref_advanced(season_end_year):
    url = f"https://www.basketball-reference.com/leagues/NBA_{season_end_year}_advanced.html"
    print(f"Pulling advanced {season_end_year-1}-{str(season_end_year)[-2:]}...")
    
    headers = {"User-Agent": "Mozilla/5.0"}
    resp = requests.get(url, headers=headers)
    tables = pd.read_html(resp.text)
    df = tables[0]
    
    df = df[df["Player"] != "Player"].copy()
    df = df[df["Rk"].notna()].copy()
    df = df.loc[:, ~df.columns.duplicated()]
    df = df.drop(columns=["Unnamed: 19", "Unnamed: 24"], errors="ignore")
    df["SEASON"] = f"{season_end_year-1}-{str(season_end_year)[-2:]}"
    
    skip_cols = ["Player", "Team", "Pos", "Awards", "SEASON"]
    for col in df.columns:
        if col not in skip_cols:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    
    print(f"  ✓ {len(df)} rows — Team sample: {df['Team'].dropna().head(3).tolist()}")
    time.sleep(4)
    return df

print("✓ Functions ready")

✓ Functions ready


## Pull all seasons for both tables

In [64]:
YEARS = [2022, 2023, 2024, 2025, 2026]

per_game_frames = []
for year in YEARS:
    df = pull_bbref_stats(year)
    per_game_frames.append(df)

bbref_df = pd.concat(per_game_frames, ignore_index=True)
print(f"\n✓ Per-game total: {len(bbref_df)} rows")
print(f"Team NaN count: {bbref_df['Team'].isna().sum()}")

adv_frames = []
for year in YEARS:
    df = pull_bbref_advanced(year)
    adv_frames.append(df)

bbref_adv = pd.concat(adv_frames, ignore_index=True)
print(f"\n✓ Advanced total: {len(bbref_adv)} rows")
print(f"Team NaN count: {bbref_adv['Team'].isna().sum()}")

Pulling per-game 2021-22...
  ✓ 812 rows — Team sample: ['PHI', 'LAL', 'MIL']
Pulling per-game 2022-23...
  ✓ 679 rows — Team sample: ['PHI', 'DAL', 'POR']
Pulling per-game 2023-24...
  ✓ 735 rows — Team sample: ['PHI', 'DAL', 'MIL']
Pulling per-game 2024-25...
  ✓ 735 rows — Team sample: ['OKC', 'MIL', 'DEN']
Pulling per-game 2025-26...
  ✓ 728 rows — Team sample: ['LAL', 'OKC', 'MIN']

✓ Per-game total: 3689 rows
Team NaN count: 0
Pulling advanced 2021-22...
  ✓ 812 rows — Team sample: ['PHO', 'CHO', 'CHI']
Pulling advanced 2022-23...
  ✓ 679 rows — Team sample: ['2TM', 'PHO', 'BRK']
Pulling advanced 2023-24...
  ✓ 735 rows — Team sample: ['CHI', 'SAC', 'CHI']
Pulling advanced 2024-25...
  ✓ 735 rows — Team sample: ['NYK', 'NYK', 'MIN']
Pulling advanced 2025-26...
  ✓ 728 rows — Team sample: ['HOU', 'HOU', 'ORL']

✓ Advanced total: 3689 rows
Team NaN count: 0


In [65]:
bbref_df  = bbref_df.rename(columns={"Player": "PLAYER_NAME"})
bbref_adv = bbref_adv.rename(columns={"Player": "PLAYER_NAME"})

# Get last team BEFORE dedup
stints = bbref_df[bbref_df["Team"] != "TOT"].copy()
last_teams = (
    stints
    .groupby(["PLAYER_NAME", "SEASON"])
    .last()
    .reset_index()[["PLAYER_NAME", "SEASON", "Team"]]
    .rename(columns={"Team": "LAST_TEAM"})
)

# Clean traded players
bbref_df  = handle_traded_players(bbref_df)
bbref_adv = handle_traded_players(bbref_adv)

print(f"\n✓ Team NaN after clean: {bbref_df['Team'].isna().sum()}")
print(bbref_df[["PLAYER_NAME", "Team", "SEASON"]].head(5))

  ✓ Traded players:   0
  ✓ Rows after clean: 3689
  ✓ Traded players:   0
  ✓ Rows after clean: 3689

✓ Team NaN after clean: 0
             PLAYER_NAME Team   SEASON
0            Joel Embiid  PHI  2021-22
1           LeBron James  LAL  2021-22
2  Giannis Antetokounmpo  MIL  2021-22
3           Kevin Durant  BRK  2021-22
4          Luka DonÄiÄ  DAL  2021-22


## Define traded player functions

In [66]:
def handle_traded_players(df):
    # BBRef uses "Tm" column — rename to TEAM if needed
    if "Tm" in df.columns and "Team" not in df.columns:
        df = df.rename(columns={"Tm": "Team"})
    if "Player" in df.columns and "PLAYER_NAME" not in df.columns:
        df = df.rename(columns={"Player": "PLAYER_NAME"})
        
    tot_players = df[df["Team"] == "TOT"][["PLAYER_NAME", "SEASON"]].drop_duplicates()
    tot_players["HAS_TOT"] = True
    df = df.merge(tot_players, on=["PLAYER_NAME", "SEASON"], how="left")
    
    df_clean = df[
        (df["HAS_TOT"] != True) |
        (df["Team"] == "TOT")
    ].copy()
    
    df_clean = df_clean.drop(columns=["HAS_TOT"])
    
    print(f"  ✓ Traded players:   {len(tot_players)}")
    print(f"  ✓ Rows after clean: {len(df_clean)}")
    return df_clean.reset_index(drop=True)

def get_last_team(df_raw):
    # Work on raw data before TOT dedup
    if "Tm" in df_raw.columns:
        df_raw = df_raw.rename(columns={"Tm": "Team"})
    if "Player" in df_raw.columns:
        df_raw = df_raw.rename(columns={"Player": "PLAYER_NAME"})
        
    stints = df_raw[df_raw["Team"] != "TOT"].copy()
    last_team = (stints
                 .groupby(["PLAYER_NAME", "SEASON"])
                 .last()
                 .reset_index()[["PLAYER_NAME", "SEASON", "Team"]]
                 .rename(columns={"Team": "LAST_TEAM"}))
    return last_team

print("✓ Traded player functions defined")

✓ Traded player functions defined


In [67]:
df.head()

,Rk,Player,Age,Team,Pos,G,GS,MP,PER,TS%,3PAr,FTr,ORB%,DRB%,TRB%,AST%,STL%,BLK%,TOV%,USG%,OWS,DWS,WS,WS/48,OBPM,DBPM,BPM,VORP,Awards,SEASON
0,1.0,Amen Thompson,23.0,HOU,PG,78.0,78.0,2911.0,18.6,0.589,0.113,0.375,8.9,14.0,11.5,20.4,2.0,1.5,13.6,19.9,6.1,3.8,9.9,0.163,1.4,1.0,2.5,3.3,NaN,2025-26
1,2.0,Kevin Durant,37.0,HOU,SF,77.0,77.0,2801.0,21.0,0.639,0.331,0.337,1.6,14.7,8.3,20.7,1.1,2.3,13.5,27.1,7.2,3.2,10.4,0.179,4.3,0.1,4.4,4.5,AS,2025-26
2,3.0,Desmond Bane,27.0,ORL,SG,80.0,80.0,2721.0,17.0,0.609,0.355,0.288,3.9,9.9,6.9,19.3,1.5,1.2,10.8,23.4,4.9,2.4,7.3,0.129,1.7,-0.3,1.4,2.3,NaN,2025-26
3,4.0,Jabari Smith Jr.,22.0,HOU,PF,76.0,76.0,2666.0,13.5,0.570,0.501,0.211,4.5,17.0,10.9,7.4,1.1,2.4,9.0,18.3,3.3,3.2,6.6,0.118,-0.6,-0.1,-0.7,0.9,NaN,2025-26
4,5.0,Toumani Camara,25.0,POR,PF,80.0,80.0,2665.0,11.5,0.584,0.667,0.159,5.1,11.9,8.5,10.5,1.7,1.1,13.5,16.3,2.1,2.6,4.7,0.084,-0.9,0.0,-0.9,0.7,NaN,2025-26


## Capture last team BEFORE dedup, then clean both

In [68]:
# Must get last team BEFORE we remove individual team rows
last_teams = get_last_team(bbref_df)

# Now clean both
print("Cleaning per-game...")
bbref_df = handle_traded_players(bbref_df)

print("\nCleaning advanced...")
bbref_adv = handle_traded_players(bbref_adv)

print("\n✓ Both DataFrames clean")

Cleaning per-game...
  ✓ Traded players:   0
  ✓ Rows after clean: 3689

Cleaning advanced...
  ✓ Traded players:   0
  ✓ Rows after clean: 3689

✓ Both DataFrames clean


## Merge per-game + advanced

In [69]:
adv_keep = ["PLAYER_NAME", "Team", "SEASON",
            "PER", "TS%", "3PAr", "FTr", "ORB%", "DRB%", "TRB%",
            "AST%", "STL%", "BLK%", "TOV%", "USG%",
            "OWS", "DWS", "WS", "WS/48", "OBPM", "DBPM", "BPM", "VORP"]

adv_keep = [c for c in adv_keep if c in bbref_adv.columns]

master = bbref_df.merge(
    bbref_adv[adv_keep],
    on=["PLAYER_NAME", "Team", "SEASON"],
    how="left"
)

# Add last team
master = master.merge(last_teams, on=["PLAYER_NAME", "SEASON"], how="left")
master["LAST_TEAM"] = master["LAST_TEAM"].fillna(master["Team"])

print(f"✓ Master shape: {master.shape}")
master.head()

✓ Master shape: (3689, 53)


,Rk,PLAYER_NAME,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards,SEASON,PER,TS%,3PAr,FTr,ORB%,DRB%,TRB%,AST%,STL%,BLK%,TOV%,USG%,OWS,DWS,WS,WS/48,OBPM,DBPM,BPM,VORP,LAST_TEAM
0,1.0,Joel Embiid,27.0,PHI,C,68.0,68.0,33.8,9.8,19.6,0.499,1.4,3.7,0.371,8.4,15.9,0.529,0.534,9.6,11.8,0.814,2.1,9.6,11.7,4.2,1.1,1.5,3.1,2.7,30.6,"MVP-2,AS,NBA2",2021-22,31.2,0.616,0.188,0.602,7.2,31.1,19.4,23.5,1.7,3.9,11.3,37.2,7.9,4.1,12.0,0.252,7.2,2.0,9.2,6.5,PHI
1,2.0,LeBron James,37.0,LAL,C,56.0,56.0,37.2,11.4,21.8,0.524,2.9,8.0,0.359,8.6,13.8,0.620,0.590,4.5,6.0,0.756,1.1,7.1,8.2,6.2,1.3,1.1,3.5,2.2,30.3,"MVP-10,AS,NBA3",2021-22,26.2,0.619,0.367,0.275,3.3,20.4,11.8,30.6,1.7,2.5,12.5,32.3,5.2,2.3,7.5,0.172,6.9,0.8,7.7,5.1,LAL
2,3.0,Giannis Antetokounmpo,27.0,MIL,PF,67.0,67.0,32.9,10.3,18.6,0.553,1.1,3.6,0.293,9.2,15.0,0.616,0.582,8.3,11.4,0.722,2.0,9.6,11.6,5.8,1.1,1.4,3.3,3.2,29.9,"MVP-3,DPOY-6,AS,NBA1,DEF1",2021-22,32.1,0.633,0.194,0.615,6.6,30.4,18.7,31.7,1.6,4.0,12.2,34.9,9.2,3.7,12.9,0.281,7.6,3.5,11.2,7.4,MIL
3,4.0,Kevin Durant,33.0,BRK,PF,55.0,55.0,37.2,10.5,20.3,0.518,2.1,5.5,0.383,8.4,14.8,0.568,0.570,6.8,7.4,0.910,0.5,6.9,7.4,6.4,0.9,0.9,3.5,2.1,29.9,"MVP-10,AS,NBA2",2021-22,25.6,0.634,0.269,0.367,1.6,19.6,10.8,29.1,1.1,2.3,12.9,31.2,6.4,2.0,8.4,0.198,6.4,0.7,7.2,4.8,BRK
4,5.0,Luka DonÄiÄ,22.0,DAL,PG,65.0,65.0,35.4,9.9,21.6,0.457,3.1,8.8,0.353,6.8,12.8,0.528,0.529,5.6,7.5,0.744,0.9,8.3,9.1,8.7,1.2,0.6,4.5,2.2,28.4,"MVP-5,AS,NBA1",2021-22,25.1,0.571,0.406,0.349,2.7,26.0,14.3,46.0,1.6,1.4,15.3,37.4,3.8,3.8,7.6,0.159,6.4,1.8,8.2,5.9,DAL


## Filter and tag

In [70]:
master = master[master["G"] >= 20].copy()

master["DATA_TYPE"] = master["SEASON"].apply(
    lambda s: "current" if s in ["2024-25", "2025-26"] else "training"
)

master = master.reset_index(drop=True)

print(f"✓ Final shape: {master.shape}")
print(f"\nBreakdown:")
print(master.groupby(["SEASON","DATA_TYPE"])["PLAYER_NAME"].count().to_string())

✓ Final shape: (2663, 54)

Breakdown:
SEASON   DATA_TYPE
2021-22  training     528
2022-23  training     528
2023-24  training     532
2024-25  current      543
2025-26  current      532


## Sanity Check/Additional Cleaning

In [71]:
check_cols = ["PLAYER_NAME", "Team", "SEASON", "G", 
              "PTS", "AST", "TRB", "BPM", "VORP", "WS"]

print("Top 10 by BPM — 2025-26:\n")
(master[master["SEASON"] == "2025-26"]
 [check_cols]
 .sort_values("BPM", ascending=False)
 .head(10)
 .reset_index(drop=True))

Top 10 by BPM — 2025-26:



,PLAYER_NAME,Team,SEASON,G,PTS,AST,TRB,BPM,VORP,WS
0,Nikola JokiÄ,DEN,2025-26,64.0,27.8,10.9,12.9,14.1,9.0,14.6
1,Shai Gilgeous-Alexander,OKC,2025-26,68.0,31.1,6.6,4.3,11.7,7.8,15.4
2,Victor Wembanyama,SAS,2025-26,63.0,24.8,3.1,11.5,10.5,5.8,9.7
3,Giannis Antetokounmpo,MIL,2025-26,36.0,27.6,5.4,9.8,9.5,3.0,5.0
4,Luka DonÄiÄ,LAL,2025-26,64.0,33.5,8.3,7.7,9.2,6.5,9.4
5,Kawhi Leonard,LAC,2025-26,64.0,28.0,3.6,6.3,8.0,5.2,9.1
6,Cade Cunningham,DET,2025-26,62.0,24.4,9.9,5.6,6.5,4.6,7.8
7,Jimmy Butler,GSW,2025-26,38.0,20.0,4.9,5.6,5.6,2.3,5.8
8,Tyrese Maxey,PHI,2025-26,68.0,28.3,6.7,4.1,5.4,4.8,8.4
9,Stephen Curry,GSW,2025-26,41.0,27.0,4.8,3.5,5.4,2.4,4.0


In [79]:
# Full cleaning cell — run this before saving

# 1. Standardize column names to uppercase
master.columns = [c.upper().replace("/","_").replace("%","_PCT").replace("-","_") 
                  for c in master.columns]

# 2. Rename key columns to cleaner names
master = master.rename(columns={
    "TRB"   : "REB",
    "MP"    : "MIN",
    "GS"    : "GAMES_STARTED",
    "RK"    : "RANK",
    "AGE"   : "AGE",
    "POS"   : "POSITION",
})

# 3. Standardize position values
pos_map = {
    "PG"    : "G",
    "SG"    : "G", 
    "SF"    : "F",
    "PF"    : "F",
    "C"     : "C",
    "PG-SG" : "G",
    "SG-PG" : "G",
    "SF-PF" : "F",
    "PF-SF" : "F",
    "PF-C"  : "F",
    "C-PF"  : "C",
    "SG-SF" : "G",
    "SF-SG" : "F",
}
master["POSITION_GROUP"] = master["POSITION"].map(pos_map).fillna("F")

# 4. Strip whitespace from all string columns
str_cols = master.select_dtypes(include="object").columns
for col in str_cols:
    master[col] = master[col].str.strip()

# 5. Drop the Rk column (meaningless after concat)
master = master.drop(columns=["RANK"], errors="ignore")

# 6. Add age as integer
master["AGE"] = master["AGE"].astype(float).astype("Int64")

# 7. Final dedup safety check
before = len(master)
master = master.drop_duplicates(
    subset=["PLAYER_NAME", "SEASON", "TEAM"],
    keep="first"
).reset_index(drop=True)
print(f"Dedup removed: {before - len(master)} rows")

# 8. Summary
print(f"\n✓ Final clean shape: {master.shape}")
print(f"Seasons: {sorted(master['SEASON'].unique())}")
print(f"Null counts in key columns:")
print(master[["PLAYER_NAME","TEAM","POSITION","AGE","BPM","VORP","WS"]].isna().sum())

Dedup removed: 0 rows

✓ Final clean shape: (2663, 55)
Seasons: ['2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
Null counts in key columns:
PLAYER_NAME    0
TEAM           0
POSITION       0
AGE            0
BPM            0
VORP           0
WS             0
dtype: int64


In [85]:
name_fixes_exact = {
    # Broken unicode — Eastern European names
    "Nikola JokiA"       : "Nikola Jokic",
    "Luka DonAiA"        : "Luka Doncic",
    "Bogdan BogdanoviA"  : "Bogdan Bogdanovic",
    "Bojan BogdanoviA"   : "Bojan Bogdanovic",
    "Boban MarjanoviA"   : "Boban Marjanovic",
    "Goran DragiA"       : "Goran Dragic",
    "Jusuf NurkiA"       : "Jusuf Nurkic",
    "Nikola JoviA"       : "Nikola Jovic",
    "Nikola VuAeviA"     : "Nikola Vucevic",
    "Vasilije MiciA"     : "Vasilije Micic",
    "Karlo MatkoviA"     : "Karlo Matkovic",
    "Kristaps PorziAAis" : "Kristaps Porzingis",
    "Dario A ariA"       : "Dario Saric",
    "Luka A amaniA"      : "Luka Samanic",
    "DAvis BertAns"      : "Davis Bertans",
    "Egor DNmin"         : "Egor Demin",
    "TomAA SatoranskA12" : "Tomas Satoransky",
    "VAt KrejAA"         : "Vit Krejci",
}

# Apply fixes
master["PLAYER_NAME"] = master["PLAYER_NAME"].replace(name_fixes_exact)

# Verify — these should all show clean names now
verify = ["Joki", "DonA", "Bogdan", "Porzi", "Nurkic", "Demin"]
for v in verify:
    names = master[master["PLAYER_NAME"].str.contains(v, na=False)]["PLAYER_NAME"].unique()
    if len(names) > 0:
        print(f"{v}: {names}")

# Final check — anything still broken?
still_broken = master[
    master["PLAYER_NAME"].str.contains(r'A\b.*A\b|AAis|DNmin|AariA', na=False, regex=True)
]["PLAYER_NAME"].unique()

print(f"\nStill broken: {len(still_broken)}")
for n in still_broken:
    print(f"  {repr(n)}")

Joki: ['Nikola Jokic']
Bogdan: ['Bojan Bogdanovic' 'Bogdan Bogdanovic']
Porzi: ['Kristaps Porzingis']
Nurkic: ['Jusuf Nurkic']
Demin: ['Egor Demin']

Still broken: 0


In [86]:
def fix_player_names(name):
    """Convert accented characters to ASCII equivalents."""
    if pd.isna(name):
        return name
    # Normalize unicode — converts Jokić → Jokic, Dončić → Doncic
    return unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode("utf-8").strip()

# Apply name fix
master["PLAYER_NAME_RAW"] = master["PLAYER_NAME"]  # keep original just in case
master["PLAYER_NAME"] = master["PLAYER_NAME"].apply(fix_player_names)

# Verify the problematic names
print("Name fix check:")
print(master[master["PLAYER_NAME"].str.contains("Joki|Donc", na=False)][
    ["PLAYER_NAME_RAW", "PLAYER_NAME", "TEAM", "SEASON"]
].head(6))

Name fix check:
     PLAYER_NAME_RAW   PLAYER_NAME TEAM   SEASON
4        Luka Doncic   Luka Doncic  DAL  2021-22
9       Nikola Jokic  Nikola Jokic  DEN  2021-22
529      Luka Doncic   Luka Doncic  DAL  2022-23
555     Nikola Jokic  Nikola Jokic  DEN  2022-23
1057     Luka Doncic   Luka Doncic  DAL  2023-24
1067    Nikola Jokic  Nikola Jokic  DEN  2023-24


## Save

In [87]:
# Update raw backup to match
master["PLAYER_NAME_RAW"] = master["PLAYER_NAME_RAW"].replace(name_fixes_exact)

# Resave
master.to_csv("../data/raw/bbref_master.csv", index=False)
print(f"✓ Saved clean version — {len(master)} rows")

# Final sanity check
print("\nTop 10 by BPM 2025-26:")
print(master[master["SEASON"] == "2025-26"][
    ["PLAYER_NAME", "TEAM", "BPM", "VORP", "WS"]
].sort_values("BPM", ascending=False).head(10).to_string())

✓ Saved clean version — 2663 rows

Top 10 by BPM 2025-26:
                  PLAYER_NAME TEAM   BPM  VORP    WS
2138             Nikola Jokic  DEN  14.1   9.0  14.6
2132  Shai Gilgeous-Alexander  OKC  11.7   7.8  15.4
2147        Victor Wembanyama  SAS  10.5   5.8   9.7
2139    Giannis Antetokounmpo  MIL   9.5   3.0   5.0
2131              Luka Doncic  LAL   9.2   6.5   9.4
2136            Kawhi Leonard  LAC   8.0   5.2   9.1
2148          Cade Cunningham  DET   6.5   4.6   7.8
2177             Jimmy Butler  GSW   5.6   2.3   5.8
2135             Tyrese Maxey  PHI   5.4   4.8   8.4
2140            Stephen Curry  GSW   5.4   2.4   4.0
